# S6.8 — Specialized Agents

Interactive verification of the four specialized agents:
- **SummarizerAgent**: Structured paper summaries
- **FactCheckerAgent**: Claim verification with evidence
- **TrendAnalyzerAgent**: Research trend identification
- **CitationTrackerAgent**: Citation relationship mapping
- **AgentRegistry**: Dispatch mechanism

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.agents.specialized import (
    AgentRegistry,
    CitationTrackerAgent,
    FactCheckerAgent,
    SummarizerAgent,
    TrendAnalyzerAgent,
)
from src.services.agents.specialized.base import SpecializedAgentBase, SpecializedAgentResult
from src.services.agents.models import SourceItem
from src.services.agents.context import AgentContext

print("✓ All specialized agent imports successful")

✓ All specialized agent imports successful


## FR-1: Base Protocol

In [2]:
# SpecializedAgentBase is abstract — cannot be instantiated
try:
    SpecializedAgentBase()
    assert False, "Should have raised TypeError"
except TypeError:
    print("✓ SpecializedAgentBase is abstract (cannot instantiate)")

# SpecializedAgentResult has correct fields and defaults
result = SpecializedAgentResult(agent_name="test", analysis="Test analysis")
assert result.agent_name == "test"
assert result.analysis == "Test analysis"
assert result.sources == []
assert result.metadata == {}
print("✓ SpecializedAgentResult has correct fields and defaults")

✓ SpecializedAgentBase is abstract (cannot instantiate)
✓ SpecializedAgentResult has correct fields and defaults


## FR-2 to FR-5: All Four Agents

In [3]:
# Verify all 4 agents implement the protocol
agents = [SummarizerAgent(), FactCheckerAgent(), TrendAnalyzerAgent(), CitationTrackerAgent()]

for agent in agents:
    assert isinstance(agent, SpecializedAgentBase), f"{agent.name} must extend SpecializedAgentBase"
    assert hasattr(agent, "run"), f"{agent.name} must have run()"
    assert hasattr(agent, "name"), f"{agent.name} must have name property"
    assert isinstance(agent.name, str) and len(agent.name) > 0
    print(f"  ✓ {agent.name}: implements protocol, has run() and name")

print("\n✓ All 4 specialized agents implement SpecializedAgentBase")

  ✓ summarizer: implements protocol, has run() and name
  ✓ fact_checker: implements protocol, has run() and name
  ✓ trend_analyzer: implements protocol, has run() and name
  ✓ citation_tracker: implements protocol, has run() and name

✓ All 4 specialized agents implement SpecializedAgentBase


In [4]:
from unittest.mock import AsyncMock, MagicMock
from src.services.agents.specialized.summarizer import SummarizerResult
from src.services.agents.specialized.fact_checker import FactCheckResult, ClaimVerification
from src.services.agents.specialized.trend_analyzer import TrendAnalysisResult, TrendItem
from src.services.agents.specialized.citation_tracker import CitationTrackResult

# Create sample papers
sample_sources = [
    SourceItem(
        arxiv_id="1706.03762",
        title="Attention Is All You Need",
        authors=["Vaswani", "Shazeer", "Parmar"],
        url="https://arxiv.org/abs/1706.03762",
        relevance_score=0.95,
        chunk_text="We propose a new architecture based entirely on attention mechanisms...",
    ),
    SourceItem(
        arxiv_id="1810.04805",
        title="BERT: Pre-training of Deep Bidirectional Transformers",
        authors=["Devlin", "Chang", "Lee", "Toutanova"],
        url="https://arxiv.org/abs/1810.04805",
        relevance_score=0.88,
        chunk_text="We introduce BERT, a new language representation model...",
    ),
]

print("✓ Sample papers and result models ready")

✓ Sample papers and result models ready


In [5]:
# Helper to mock LLM for structured output
def make_mock_context(return_value):
    mock_provider = MagicMock()
    mock_model = MagicMock()
    mock_structured = MagicMock()
    mock_structured.ainvoke = AsyncMock(return_value=return_value)
    mock_model.with_structured_output.return_value = mock_structured
    mock_provider.get_langchain_model.return_value = mock_model
    return AgentContext(llm_provider=mock_provider, model_name="test-model")

# --- SummarizerAgent ---
ctx = make_mock_context(SummarizerResult(
    objective="Study self-attention for sequence transduction.",
    methodology="Replace recurrence with multi-head attention.",
    key_findings=["Outperforms RNNs on translation"],
    contributions=["Transformer architecture"],
    limitations=["Quadratic memory"],
    summary_text="The paper proposes the Transformer...",
    sources=sample_sources,
))
result = await SummarizerAgent().run("Summarize Attention Is All You Need", ctx, papers=sample_sources)
assert isinstance(result, SummarizerResult)
assert result.objective != ""
assert len(result.key_findings) > 0
assert len(result.sources) > 0
print(f"✓ SummarizerAgent: objective='{result.objective[:50]}...', {len(result.key_findings)} findings, {len(result.sources)} sources")

# --- FactCheckerAgent ---
ctx = make_mock_context(FactCheckResult(
    claims=[ClaimVerification(claim="Transformers beat RNNs", verdict="supported", evidence=[sample_sources[0]], explanation="Evidence in Vaswani et al.")],
    sources=sample_sources[:1],
    overall_assessment="Supported by literature.",
))
result = await FactCheckerAgent().run("Verify: Transformers beat RNNs", ctx, papers=sample_sources)
assert isinstance(result, FactCheckResult)
assert result.claims[0].verdict == "supported"
print(f"✓ FactCheckerAgent: claim='{result.claims[0].claim}', verdict={result.claims[0].verdict}")

# --- TrendAnalyzerAgent ---
ctx = make_mock_context(TrendAnalysisResult(
    trends=[TrendItem(topic="Transformers", direction="rising", key_papers=[sample_sources[0]], description="Dominant architecture.")],
    timeline="2017: Transformer. 2018: BERT.",
    emerging_topics=["Efficient transformers"],
    sources=sample_sources,
))
result = await TrendAnalyzerAgent().run("NLP trends", ctx, papers=sample_sources)
assert isinstance(result, TrendAnalysisResult)
assert result.trends[0].direction == "rising"
print(f"✓ TrendAnalyzerAgent: {len(result.trends)} trends, {len(result.emerging_topics)} emerging topics")

# --- CitationTrackerAgent ---
ctx = make_mock_context(CitationTrackResult(
    target_paper=sample_sources[0],
    cited_by=[sample_sources[1]],
    references=[],
    citation_count=1,
    influence_summary="Cited by BERT.",
    sources=sample_sources,
))
result = await CitationTrackerAgent().run("Citations for Attention Is All You Need", ctx, papers=sample_sources)
assert isinstance(result, CitationTrackResult)
assert result.citation_count == 1
print(f"✓ CitationTrackerAgent: target='{result.target_paper.title}', cited_by={result.citation_count}")

print("\n✓ All 4 specialized agents produce correct structured output")

✓ SummarizerAgent: objective='Study self-attention for sequence transduction....', 1 findings, 2 sources
✓ FactCheckerAgent: claim='Transformers beat RNNs', verdict=supported
✓ TrendAnalyzerAgent: 1 trends, 1 emerging topics
✓ CitationTrackerAgent: target='Attention Is All You Need', cited_by=1

✓ All 4 specialized agents produce correct structured output


## FR-6: Agent Registry

In [6]:
registry = AgentRegistry()

# All 4 agents registered
assert len(registry.agent_names) == 4
print(f"Registered agents: {registry.agent_names}")

# Dispatch to correct type
assert isinstance(registry.get("summarizer"), SummarizerAgent)
assert isinstance(registry.get("fact_checker"), FactCheckerAgent)
assert isinstance(registry.get("trend_analyzer"), TrendAnalyzerAgent)
assert isinstance(registry.get("citation_tracker"), CitationTrackerAgent)
print("✓ All agents dispatch to correct type")

# Unknown agent returns None
assert registry.get("nonexistent") is None
print("✓ Unknown agent returns None")

print("\n✓ AgentRegistry dispatches correctly and handles unknown types")

Registered agents: ['summarizer', 'fact_checker', 'trend_analyzer', 'citation_tracker']
✓ All agents dispatch to correct type
✓ Unknown agent returns None

✓ AgentRegistry dispatches correctly and handles unknown types


## Source Citation Verification

In [7]:
# Verify all agent results include arXiv links in sources
for source in sample_sources:
    assert "arxiv.org" in source.url, f"Missing arXiv link: {source.url}"

# Verify result models accept sources
summary = SummarizerResult(objective="t", methodology="t", key_findings=["t"], contributions=["t"], limitations=["t"], summary_text="t", sources=sample_sources)
assert all("arxiv.org" in s.url for s in summary.sources)

fact = FactCheckResult(claims=[], sources=sample_sources, overall_assessment="t")
assert all("arxiv.org" in s.url for s in fact.sources)

trend = TrendAnalysisResult(trends=[], timeline="t", emerging_topics=[], sources=sample_sources)
assert all("arxiv.org" in s.url for s in trend.sources)

cite = CitationTrackResult(target_paper=sample_sources[0], sources=sample_sources, citation_count=0, influence_summary="t")
assert all("arxiv.org" in s.url for s in cite.sources)

print("✓ All result models carry sources with arXiv links")

✓ All result models carry sources with arXiv links
